# Setup

## Import modules

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import torch

# Import pipeline modules
from scoring import score_image
from selection import select
from metrics import MetricsTracker
from generation.UnconditionalGenerator import UnconditionalGenerator
from utils import save_image, set_seed

# Config
OUTPUT_DIR = Path("outputs")
IMAGES_DIR = OUTPUT_DIR / "images"
METRICS_CSV = OUTPUT_DIR / "metrics.csv"

# Experiment parameters
NUM_PROMPTS = 2
NUM_TRIALS = 1
ROUNDS = 2        # r
BATCH_SIZE = 4    # B
SEED = 42

# Chosen strategies
SELECTION_STRATEGY = "argmax"

# Baseline scoring rubric (example)
# CLIP_RUBRIC = {"type": "clip", "text": "A photorealistic portrait of a dog", "weight": 1.0}
BRIGHTNESS_RUBRIC = {"type": "brightness", "weight": 1.0}

set_seed(SEED)


## Load diffusion model

In [2]:
from diffusers import StableDiffusionPipeline

MODEL_NAME = "stabilityai/stable-diffusion-2-1"  # TODO: replace with SD3 HF model if available
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.bfloat16
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # memory-friendly

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


## Load dataset

In [3]:
# Mock dataset
dataset = [
    {"prompt_id": f"p{i}+1", 
    "prompt": f"A photograph of {i+1} dogs",
    "rubric": BRIGHTNESS_RUBRIC}
    for i in range(NUM_PROMPTS)
]

# Run simulation

In [ ]:
metrics = MetricsTracker()
policy = UnconditionalGenerator(pipe)

for trial in range(NUM_TRIALS):
    print("="*100)
    print(f"BEGINNING TRIAL {trial}")
    print("="*100)

    trial_id = f"trial{trial}"

    for item in tqdm(dataset, desc="prompts"):
        prompt_id = item["prompt_id"]
        prompt = item["prompt"]
        rubric = item["rubric"]

        print(f"Item: prompt_id={prompt_id}, prompt={prompt}, rubric={rubric}")

        prompt_outdir = IMAGES_DIR / prompt_id / trial_id
        prompt_outdir.mkdir(parents=True, exist_ok=True)

        for round in range(ROUNDS):
            print(f"\tRound {round}")
            round_outdir = prompt_outdir / f"round{round}"

            # Generate images using our method
            batch_images: list[Image.Image] = policy.generate(
                prompt,
                sampling_parameters={
                    "num_images_per_prompt": BATCH_SIZE,
                    "num_inference_steps": 5,
                    "guidance_scale": 7.5,
                    "width": 512,
                    "height": 512,
                    # "generator": torch.Generator(device="cuda").manual_seed(SEED), # this leads to different trials having the same images
                }
            )
            
            # Score and record metrics for each image in the batch
            user_scores = []
            for image_idx, image in enumerate(batch_images):
                image_path = round_outdir / f"image{image_idx}.png"
                
                user_score = score_image(image, rubric)
                user_scores.append(user_score)
                
                metrics.log(
                    prompt_id,
                    trial_id,
                    round,
                    image_idx,
                    image_path,
                    user_score,
                    image,
                    extra_info={
                        "prompt": prompt,
                    }
                )
                save_image(image, image_path)

            # Select favorite (index)
            chosen_image_idx = select(user_scores, strategy=SELECTION_STRATEGY)
            metrics.mark_chosen(
                prompt_id,
                trial_id,
                round,
                chosen_image_idx,
            )

            # Update policy
            policy.update(
                feedback={
                    # nothing for now
                }
            )

            print(f"\t\tScores: {user_scores}")
            print(f"\t\tChosen index (using {SELECTION_STRATEGY}): {chosen_image_idx}")
            
            
# Save metrics
metrics.save_csv(str(METRICS_CSV))
print("Saved metrics to", METRICS_CSV)

BEGINNING TRIAL 0


prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Item: prompt_id=p0+1, prompt=A photograph of 1 dogs, rubric={'type': 'brightness', 'weight': 1.0}
	Round 0


  0%|          | 0/5 [00:00<?, ?it/s]

		Scores: [0.4201687276363373, 0.4297652542591095, 0.45912304520606995, 0.41806164383888245]
		Chosen index (using argmax): 2
	Round 1


...

  0%|          | 0/5 [00:00<?, ?it/s]

		Scores: [0.4209728240966797, 0.43228933215141296, 0.4701392352581024, 0.4100781977176666]
		Chosen index (using argmax): 2
	Round 1


  0%|          | 0/5 [00:00<?, ?it/s]

		Scores: [0.4209728240966797, 0.43228933215141296, 0.4701392352581024, 0.4100781977176666]
		Chosen index (using argmax): 2


AttributeError: 'MetricsTracker' object has no attribute 'save_csv'

In [ ]:
batch_images